# Chapter 02 — Micrograd Value Class (Simplified)

Every modern neural network learns through backpropagation — the algorithm that computes how much each weight contributed to the model's error. But most practitioners treat it as a black box. In this chapter, you will build an automatic differentiation engine from scratch, giving you deep intuition about how neural networks actually learn.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

# Demo: gradient computation
a = Value(2.0)
b = Value(3.0)
c = a * b  # c = 6.0
c.backward()
print(f"c = a * b = {c.data}")
print(f"dc/da = {a.grad}")  # Expected: 3.0 (= b.data)
print(f"dc/db = {b.grad}")  # Expected: 2.0 (= a.data)

---

**Course:** Aprenda LLM | **Chapter 02**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.